In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

bronze_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/carriers/"
)

silver_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/carriers/"
)

dq_invalid_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "quality/dq_invalid_records/carriers/"
)

dq_metrics_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "quality/dq_metrics/carriers/"
)

# ============================================================
# 2. LECTURE
# ============================================================

bronze_carriers = (
    spark.read
    .format("delta")
    .load(bronze_carriers_path)
)

print(
    "Nombre de carriers Bronze :",
    bronze_carriers.count()
)

# ============================================================
# 3. NORMALISATION
# ============================================================

normalized_carriers = (
    bronze_carriers

    .withColumn(
        "carrier_id",
        F.upper(
            F.trim(
                F.col("carrier_id")
            )
        )
    )

    .withColumn(
        "carrier_name",
        F.trim(
            F.col("carrier_name")
        )
    )

    .withColumn(
        "carrier_type",
        F.initcap(
            F.trim(
                F.col("carrier_type")
            )
        )
    )

    .withColumn(
        "country",
        F.initcap(
            F.trim(
                F.col("country")
            )
        )
    )

    .withColumn(
        "service_level",
        F.initcap(
            F.trim(
                F.col("service_level")
            )
        )
    )
)

# ============================================================
# 4. DÉDUPLICATION
# ============================================================

dedup_window = (
    Window
    .partitionBy("carrier_id")
    .orderBy(
        F.col("updated_at").desc_nulls_last(),
        F.col("ingestion_timestamp").desc_nulls_last()
    )
)

carriers_ranked = (
    normalized_carriers
    .withColumn(
        "_row_number",
        F.row_number().over(dedup_window)
    )
)

duplicate_carriers = (
    carriers_ranked
    .filter(
        F.col("_row_number") > 1
    )
)

deduplicated_carriers = (
    carriers_ranked
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

# ============================================================
# 5. RÈGLES DATA QUALITY
# ============================================================

allowed_carrier_types = [
    "Express",
    "Standard",
    "Economy"
]

allowed_service_levels = [
    "Premium",
    "Standard",
    "Economy"
]

validated_carriers = (
    deduplicated_carriers

    .withColumn(
        "dq_carrier_id_null",
        F.col("carrier_id").isNull()
        | (F.col("carrier_id") == "")
    )

    .withColumn(
        "dq_carrier_name_null",
        F.col("carrier_name").isNull()
        | (F.col("carrier_name") == "")
    )

    .withColumn(
        "dq_invalid_carrier_type",
        F.col("carrier_type").isNull()
        | (
            ~F.col("carrier_type")
            .isin(allowed_carrier_types)
        )
    )

    .withColumn(
        "dq_country_null",
        F.col("country").isNull()
        | (F.col("country") == "")
    )

    .withColumn(
        "dq_invalid_service_level",
        F.col("service_level").isNull()
        | (
            ~F.col("service_level")
            .isin(allowed_service_levels)
        )
    )

    .withColumn(
        "dq_active_null",
        F.col("active").isNull()
    )
)

# ============================================================
# 6. CONSTRUIRE dq_errors
# ============================================================

validated_carriers = (
    validated_carriers

    .withColumn(
        "dq_errors",
        F.array_compact(
            F.array(

                F.when(
                    F.col("dq_carrier_id_null"),
                    F.lit("CARRIER_ID_NULL")
                ),

                F.when(
                    F.col("dq_carrier_name_null"),
                    F.lit("CARRIER_NAME_NULL")
                ),

                F.when(
                    F.col("dq_invalid_carrier_type"),
                    F.lit("INVALID_CARRIER_TYPE")
                ),

                F.when(
                    F.col("dq_country_null"),
                    F.lit("COUNTRY_NULL")
                ),

                F.when(
                    F.col("dq_invalid_service_level"),
                    F.lit("INVALID_SERVICE_LEVEL")
                ),

                F.when(
                    F.col("dq_active_null"),
                    F.lit("ACTIVE_NULL")
                )
            )
        )
    )

    .withColumn(
        "dq_is_valid",
        F.size(
            F.col("dq_errors")
        ) == 0
    )
)

# ============================================================
# 7. VALIDES / INVALIDES
# ============================================================

valid_carriers = (
    validated_carriers
    .filter(
        F.col("dq_is_valid") == True
    )
)

invalid_carriers = (
    validated_carriers
    .filter(
        F.col("dq_is_valid") == False
    )
)

# ============================================================
# 8. SILVER
# ============================================================

silver_carriers = (
    valid_carriers
    .drop(
        "dq_carrier_id_null",
        "dq_carrier_name_null",
        "dq_invalid_carrier_type",
        "dq_country_null",
        "dq_invalid_service_level",
        "dq_active_null",
        "dq_errors",
        "dq_is_valid"
    )
)

# ============================================================
# 9. ÉCRITURE SILVER
# ============================================================

(
    silver_carriers
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_carriers_path)
)

# ============================================================
# 10. QUARANTAINE
# ============================================================

(
    invalid_carriers
    .write
    .format("delta")
    .mode("overwrite")
    .save(dq_invalid_carriers_path)
)

# ============================================================
# 11. MÉTRIQUES DQ
# ============================================================

records_checked = deduplicated_carriers.count()
valid_records = silver_carriers.count()
invalid_records = invalid_carriers.count()
duplicate_records = duplicate_carriers.count()

if records_checked > 0:
    dq_pass_rate = (
        valid_records
        / records_checked
    ) * 100
else:
    dq_pass_rate = 0.0

dq_metrics = [
    (
        "carriers",
        records_checked,
        valid_records,
        invalid_records,
        duplicate_records,
        float(dq_pass_rate)
    )
]

dq_metrics_schema = """
dataset STRING,
records_checked LONG,
valid_records LONG,
invalid_records LONG,
duplicate_records LONG,
dq_pass_rate DOUBLE
"""

dq_metrics_df = (
    spark.createDataFrame(
        dq_metrics,
        schema=dq_metrics_schema
    )
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
)

(
    dq_metrics_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(dq_metrics_carriers_path)
)

# ============================================================
# 12. RÉSULTATS
# ============================================================

print(
    "Bronze carriers :",
    bronze_carriers.count()
)

print(
    "Après déduplication :",
    records_checked
)

print(
    "Doublons détectés :",
    duplicate_records
)

print(
    "Carriers valides :",
    valid_records
)

print(
    "Carriers invalides :",
    invalid_records
)

print(
    "DQ pass rate :",
    round(dq_pass_rate, 2),
    "%"
)

print("\n=== SILVER CARRIERS ===")

display(
    silver_carriers
    .orderBy("carrier_id")
)

print("\n=== INVALID CARRIERS ===")

display(
    invalid_carriers.select(
        "carrier_id",
        "carrier_name",
        "carrier_type",
        "country",
        "service_level",
        "active",
        "dq_errors"
    )
)

print("\n=== DQ METRICS ===")

display(
    dq_metrics_df
)